# W05 — Trained Ranking Model vs. Baseline

**Goal:** replace the hand-weighted heuristic (W02/W04) with a model trained on real
outcomes, and validate it with precision@20 against the same staleness-only baseline
(0.31) and the hand-weighted heuristic (0.29).

**Label fix:** `trend_direction` (the W01/W02 proxy) is unreliable on near-zero baselines
— a page going from 2 to 3 clicks reads as "+50%" despite nothing meaningful happening.
This notebook replaces it with a **guarded binary label**: a page counts as "declining"
only if it had a meaningful click baseline in March AND saw a real drop into April. Same
precision@20 metric as before, so results are directly comparable to W02/W04.

**Leakage discipline:** all features come from `month=2026-03` only. The label uses
`month=2026-04` outcomes — that's a legitimate future-window label, not leakage, because
no April *feature* is ever used at prediction time (see the W03 leakage trap for what
that mistake looks like: AUC spiking toward 1.0).


## Setup

In [ ]:
!pip install duckdb huggingface_hub scikit-learn --quiet

import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

import duckdb
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

con = duckdb.connect()
con.sql("SET s3_endpoint='huggingface.co';")  # placeholder if needed by your DuckDB/HF setup


## Step 1 — Pull March features and April outcomes

Reuses the same `hf://FlyRank/internship-warehouse` access pattern as W03/W04.
Adjust table/column names below to match what W03's data contract confirmed.


In [ ]:
# March: feature snapshot
march_df = con.sql('''
    SELECT *
    FROM 'hf://datasets/FlyRank/internship-warehouse/**/*.parquet'
    WHERE month = '2026-03'
''').df()

# April: outcome snapshot (clicks only — used for the label, never as a feature)
april_outcomes = con.sql('''
    SELECT page_id, client_id, clicks AS clicks_april
    FROM 'hf://datasets/FlyRank/internship-warehouse/**/*.parquet'
    WHERE month = '2026-04'
''').df()

print(march_df.shape, april_outcomes.shape)
march_df.head()


## Step 2 — Build the guarded binary label

`is_declining` = 1 only if:
- `clicks_march >= MIN_BASELINE_CLICKS` (kills the near-zero % swing problem), AND
- `clicks_april` dropped by more than `DECLINE_THRESHOLD` relative to March

Tune `MIN_BASELINE_CLICKS` after looking at the click distribution — don't just accept
the default below without checking it against your data.


In [ ]:
MIN_BASELINE_CLICKS = 20   # TODO: sanity-check against march_df['clicks'].describe()
DECLINE_THRESHOLD = 0.15   # 15%+ drop counts as declining

df = march_df.merge(april_outcomes, on=["page_id", "client_id"], how="inner")

has_baseline = df["clicks"] >= MIN_BASELINE_CLICKS
dropped = (df["clicks"] - df["clicks_april"]) / df["clicks"] >= DECLINE_THRESHOLD

df["is_declining"] = (has_baseline & dropped).astype(int)

print("Baseline-eligible rows:", has_baseline.sum(), "/", len(df))
print("Positive label rate:", df["is_declining"].mean().round(3))


## Step 3 — Features

Start with the signals already validated in W04 (staleness, CTR-vs-position), plus
search demand from W01/W02. Add more only after confirming each one individually —
same discipline as the W04 signal-verification table.


In [ ]:
FEATURE_COLS = [
    "days_since_last_update",   # staleness
    "ctr",                       # click-through rate
    "avg_search_position",       # position — CTR-vs-position was CONFIRMED in W04
    "search_demand",             # impressions/queries — TODO: confirm exact column name from W03 contract
]

# TODO: confirm every column name against the W03 data contract before trusting this list
X = df[FEATURE_COLS].copy()
y = df["is_declining"]
groups = df["client_id"]  # for client-holdout split

X.describe()


## Step 4 — Client-holdout split

A random row split would let the model see a client's pattern in training and recognize
it again in validation — client-holdout prevents that.


In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())
assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx])), "Client leakage between splits!"


## Step 5 — Train three models

Same shape as the starter repo's reference pipeline: logistic regression (interpretable
floor), random forest (interactions), gradient boosting (best-shot performance).


In [ ]:
models = {
    "logistic_regression": LogisticRegression(max_iter=1000),
    "random_forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "gradient_boosting": GradientBoostingClassifier(random_state=42),
}

fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    print(f"{name}: trained on {len(X_train)} rows")


## Step 6 — Precision@20, compared directly to W02/W04

Same evaluation function shape as before: rank the test set by predicted probability,
take the top 20, check what fraction were truly declining.


In [ ]:
def precision_at_k(y_true, scores, k=20):
    order = np.argsort(-scores)[:k]
    return y_true.iloc[order].mean()

results = {}
for name, model in fitted.items():
    scores = model.predict_proba(X_test)[:, 1]
    results[name] = precision_at_k(y_test, scores, k=20)

# Reference points from earlier weeks
results["W02_hand_weighted_heuristic"] = 0.29
results["W04_staleness_only_baseline"] = 0.31

pd.Series(results).sort_values(ascending=False)


## Step 7 — Honest verdict

Fill this in only after looking at the actual numbers above. If nothing beats 0.31,
that is the finding — keep it, the same way W02's loss was kept. Do not tune until
something wins.

- Did any model beat the 0.31 baseline?
- Which model, and by how much?
- What does the feature importance / coefficient table say about *why*?
- What's the honest limitation of this result (sample size in test set, client mix, etc.)?


In [ ]:
# TODO: feature importance for the winning model (or all three, for comparison)
# e.g. for gradient boosting / random forest:
# pd.Series(fitted["gradient_boosting"].feature_importances_, index=FEATURE_COLS).sort_values()
